# Championship Leader Conversion: Combined Kaggle+Jolpica Analysis

## Questions

* How many times has the driver or drivers in first place at this point in the season gone on to win the drivers' championship?
* How many times has the constructor or constructors in first place at this point in the season gone on to win the constructors' championship?

This notebook answers both questions at two different definitions of "this point in the season":

1. **Kaggle-only cutoff** — the existing answer from `outputs/reports/championship_leader_conversion.md`, pinned to the latest round covered by the local `f1_results.csv` snapshot.
2. **Combined cutoff** — the true current-season cutoff, using the scoped, precedence-ruled combined Kaggle+Jolpica dataset (2026-07-09 decision, `docs/decisions.md`) to fill in genuine Kaggle gaps for the current season only.

All analysis logic below is imported from `src/f1_analytics/analysis/`; this notebook only calls reusable functions and formats their output.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from f1_analytics.config import DUCKDB_PATH
from f1_analytics.database.duckdb_io import connect_duckdb
from f1_analytics.analysis.championship_leaders import (
    build_conversion_table,
    current_leaders,
    summarize_conversion,
    format_report_date,
)
from f1_analytics.analysis.combined_championship_leaders import (
    determine_cutoff_round,
    build_combined_current_leaders,
    build_combined_conversion_tables,
    extract_current_season_deviations,
)

print(f"DuckDB path: {DUCKDB_PATH}")


DuckDB path: /Users/antonio/Documents/howdoidata/projects/f1_analytics/data/processed/f1_analytics.duckdb


## Data Sources And Cutoff Definitions

* **Kaggle Formula 1 complete dataset** — baseline source for all curated tables (`results`, `driver_standings_by_round`, `constructor_standings_by_round`, `final_driver_standings`, `final_constructor_standings`). See `docs/data_sources.md`.
* **Jolpica/Ergast API** — fully loaded, separate current-season source (`raw_jolpica_*` tables), used only for coverage comparison and, as of the 2026-07-09 decision, to fill genuine current-season gaps in the scoped combined tables below.
* **Combined tables** (`combined_results`, `combined_driver_standings_by_round`, `combined_constructor_standings_by_round`) — Kaggle wins for any season/round where Kaggle has at least one results row; Jolpica fills a season/round only where Kaggle has zero rows. Historical (pre-current-season) rows are unaffected, since Jolpica gap-fill only ever applies to the current season.
* **Kaggle-only cutoff** — the latest completed round available in local `f1_results.csv` (unchanged from the existing reports).
* **Combined cutoff** — the max round present in the combined standings tables for the target season, reflecting real current-season data instead of a stale Kaggle-only snapshot.


In [2]:
with connect_duckdb(DUCKDB_PATH, read_only=True) as conn:
    races = conn.execute("SELECT * FROM races").df()
    results = conn.execute("SELECT * FROM results").df()
    driver_standings = conn.execute("SELECT * FROM driver_standings_by_round").df()
    constructor_standings = conn.execute("SELECT * FROM constructor_standings_by_round").df()
    driver_final = conn.execute("SELECT * FROM final_driver_standings").df()
    constructor_final = conn.execute("SELECT * FROM final_constructor_standings").df()
    driver_standings_combined = conn.execute("SELECT * FROM combined_driver_standings_by_round").df()
    constructor_standings_combined = conn.execute("SELECT * FROM combined_constructor_standings_by_round").df()
    coverage = conn.execute("SELECT * FROM kaggle_jolpica_coverage").df()

target_season = int(results["season"].max())
print(f"Target season: {target_season}")


Target season: 2026


## Kaggle-Only-Cutoff Answer (Existing, For Continuity)

In [3]:
kaggle_cutoff_round = determine_cutoff_round(driver_standings, target_season)
kaggle_cutoff_race = races[(races["season"] == target_season) & (races["round"] == kaggle_cutoff_round)].iloc[0]

print(f"Kaggle-only cutoff round: {kaggle_cutoff_round}")
print(f"Cutoff race: {kaggle_cutoff_race['race_name']} ({format_report_date(kaggle_cutoff_race['date'])})")

driver_conversion_kaggle = build_conversion_table(
    driver_standings, driver_final, kaggle_cutoff_round, target_season,
    id_column="driver_id", name_column="driver_name",
)
constructor_conversion_kaggle = build_conversion_table(
    constructor_standings, constructor_final, kaggle_cutoff_round, target_season,
    id_column="constructor_id", name_column="constructor",
)
driver_summary_kaggle = summarize_conversion(driver_conversion_kaggle)
constructor_summary_kaggle = summarize_conversion(constructor_conversion_kaggle)

current_kaggle = current_leaders(driver_standings, target_season, kaggle_cutoff_round, "driver_id", "driver_name", "driver")
current_kaggle_constructor = current_leaders(
    constructor_standings, target_season, kaggle_cutoff_round, "constructor_id", "constructor", "constructor"
)

print(f"\nDrivers: leader converted to champion in {driver_summary_kaggle['converted']} of "
      f"{driver_summary_kaggle['total']} historical seasons ({driver_summary_kaggle['rate']:.1%}).")
print(f"Constructors: leader converted to champion in {constructor_summary_kaggle['converted']} of "
      f"{constructor_summary_kaggle['total']} historical seasons ({constructor_summary_kaggle['rate']:.1%}).")

print(f"\nCurrent {target_season} driver leader at round {kaggle_cutoff_round}:")
print(current_kaggle[["leader_name", "leader_points", "leader_wins"]].to_string(index=False))
print(f"\nCurrent {target_season} constructor leader at round {kaggle_cutoff_round}:")
print(current_kaggle_constructor[["leader_name", "leader_points", "leader_wins"]].to_string(index=False))


Kaggle-only cutoff round: 3
Cutoff race: Japanese Grand Prix (2026-03-29)



Drivers: leader converted to champion in 40 of 76 historical seasons (52.6%).
Constructors: leader converted to champion in 38 of 68 historical seasons (55.9%).

Current 2026 driver leader at round 3:
          leader_name  leader_points  leader_wins
Andrea Kimi Antonelli           68.0            2

Current 2026 constructor leader at round 3:
leader_name  leader_points  leader_wins
   Mercedes          123.0            3


## Combined-Cutoff Answer (Up-To-Date)

The combined cutoff reflects the true current-season point using the scoped Kaggle+Jolpica combination, instead of the stale Kaggle-only snapshot above.


In [4]:
combined_cutoff_round = determine_cutoff_round(driver_standings_combined, target_season)
print(f"Combined cutoff round: {combined_cutoff_round}")

driver_conversion_combined, constructor_conversion_combined = build_combined_conversion_tables(
    driver_standings_combined, constructor_standings_combined,
    driver_final, constructor_final, combined_cutoff_round, target_season,
)
driver_summary_combined = summarize_conversion(driver_conversion_combined)
constructor_summary_combined = summarize_conversion(constructor_conversion_combined)

combined_leaders = build_combined_current_leaders(
    driver_standings_combined, constructor_standings_combined, target_season, combined_cutoff_round
)

print(f"\nDrivers: leader converted to champion in {driver_summary_combined['converted']} of "
      f"{driver_summary_combined['total']} historical seasons with at least {combined_cutoff_round} "
      f"completed rounds ({driver_summary_combined['rate']:.1%}).")
print(f"Constructors: leader converted to champion in {constructor_summary_combined['converted']} of "
      f"{constructor_summary_combined['total']} historical seasons with at least {combined_cutoff_round} "
      f"completed rounds ({constructor_summary_combined['rate']:.1%}).")


Combined cutoff round: 9



Drivers: leader converted to champion in 50 of 69 historical seasons with at least 9 completed rounds (72.5%).
Constructors: leader converted to champion in 51 of 67 historical seasons with at least 9 completed rounds (76.1%).


### Does The Combined-Cutoff Answer Differ From The Kaggle-Only Answer?

In [5]:
cutoff_differs = kaggle_cutoff_round != combined_cutoff_round
driver_rate_differs = round(driver_summary_kaggle["rate"], 4) != round(driver_summary_combined["rate"], 4)
constructor_rate_differs = round(constructor_summary_kaggle["rate"], 4) != round(constructor_summary_combined["rate"], 4)

print(f"Cutoff round differs: {cutoff_differs} (Kaggle-only round {kaggle_cutoff_round} vs. combined round {combined_cutoff_round})")
print(f"Driver conversion rate differs: {driver_rate_differs} "
      f"({driver_summary_kaggle['rate']:.1%} vs. {driver_summary_combined['rate']:.1%})")
print(f"Constructor conversion rate differs: {constructor_rate_differs} "
      f"({constructor_summary_kaggle['rate']:.1%} vs. {constructor_summary_combined['rate']:.1%})")

print(
    "\nWhy: the Kaggle CSV snapshot only covers rounds "
    f"1-{kaggle_cutoff_round} for {target_season}, while Jolpica has real results through round "
    f"{combined_cutoff_round}. The combined cutoff uses the later round, which changes which historical "
    "seasons qualify for comparison (seasons need at least that many completed rounds) and how far into "
    "the season the current-season leader's position has been established. The leading driver is the "
    "same individual at both cutoffs in this snapshot, but their points/wins totals and the conversion "
    "rate they are compared against differ."
)


Cutoff round differs: True (Kaggle-only round 3 vs. combined round 9)
Driver conversion rate differs: True (52.6% vs. 72.5%)
Constructor conversion rate differs: True (55.9% vs. 76.1%)

Why: the Kaggle CSV snapshot only covers rounds 1-3 for 2026, while Jolpica has real results through round 9. The combined cutoff uses the later round, which changes which historical seasons qualify for comparison (seasons need at least that many completed rounds) and how far into the season the current-season leader's position has been established. The leading driver is the same individual at both cutoffs in this snapshot, but their points/wins totals and the conversion rate they are compared against differ.


## Current-Season Leaders At The Combined Cutoff

In [6]:
print(combined_leaders[["entity_type", "leader_name", "leader_points", "leader_wins"]].to_string(index=False))


entity_type           leader_name  leader_points  leader_wins
     driver Andrea Kimi Antonelli          158.0            5
constructor              Mercedes          286.0            7


## Kaggle Vs. Jolpica Deviations For The Current Season

Reused directly from the existing `kaggle_jolpica_coverage` view (no second, duplicate coverage view). Status meanings for this analysis:

* **match** — Kaggle and Jolpica report the same row count for the round; both sources agree.
* **kaggle_only** — Kaggle has rows and Jolpica does not; Kaggle is used as-is (no gap to fill).
* **jolpica_only** — Kaggle has zero rows and Jolpica has rows; this is the genuine gap the combined tables fill for the current season.
* **row_count_diverges** — both sources have rows but the counts disagree; Kaggle still wins for the combined tables, but this is surfaced here rather than silently resolved, since it means the two sources disagree on something for a round Kaggle already covers.
* **no_data** — neither source has rows for that season/round (e.g. a not-yet-run future round).


In [7]:
deviations = extract_current_season_deviations(coverage, target_season)
print(f"Deviation rows for {target_season} (status != match): {len(deviations)}")
print(deviations.to_string(index=False))

diverging = deviations[deviations["results_coverage_status"] == "row_count_diverges"]
if diverging.empty:
    print("\nNo row_count_diverges rows found for the current season: Kaggle and Jolpica agree wherever both have data.")
else:
    print(f"\n{len(diverging)} row_count_diverges round(s) found — Kaggle still wins for combined_results, but review these rounds before trusting them further:")
    print(diverging.to_string(index=False))


Deviation rows for 2026 (status != match): 6
 season  round  kaggle_result_rows  jolpica_result_rows  kaggle_qualifying_rows  jolpica_qualifying_rows results_coverage_status
   2026      4                   0                   22                       0                       22            jolpica_only
   2026      5                   0                   22                       0                       22            jolpica_only
   2026      6                   0                   22                       0                       22            jolpica_only
   2026      7                   0                   22                       0                       22            jolpica_only
   2026      8                   0                   22                       0                       22            jolpica_only
   2026      9                   0                   22                       0                       22            jolpica_only

No row_count_diverges rows found for the current se

## Limitations And Assumptions

* The combined tables are scoped to current-season leader identification only; they are additive and do not replace or modify `results`, `driver_standings_by_round`, `constructor_standings_by_round`, `final_driver_standings`, or `final_constructor_standings`.
* Jolpica only fills a season/round where Kaggle has zero rows for it (`jolpica_only` status). Any `row_count_diverges` round keeps its Kaggle rows in the combined tables and is surfaced above, not silently resolved.
* Historical (pre-target-season) conversion comparisons only include seasons with at least as many completed rounds as the cutoff being used, so the combined cutoff (a later round) can exclude some short early seasons that the Kaggle-only cutoff included.
* Jolpica driver/constructor ids were verified to match Kaggle's `drivers.driver_id` / `constructors.constructor_id` directly at the row level for the current season's gap rounds (all ids matched with no reconciliation needed); this was not previously verified beyond aggregate row counts.
* Points systems, sprint formats, and season lengths vary across eras; conversion rates are useful for narrative comparison but are not normalized for those differences.

## Follow-Up Directions

* Re-run this notebook as Kaggle's snapshot catches up, to see the Kaggle-only and combined cutoffs converge.
* Extend the combined-dataset precedence rule to qualifying data if a current-season qualifying-based analysis is ever needed.
* Investigate any `row_count_diverges` rounds found above before relying on them for anything beyond leader identification.

## Output Tables Referenced

* `outputs/reports/championship_leader_conversion.md` (existing Kaggle-only report, unchanged)
* `outputs/reports/championship_leader_followup.md` (existing Kaggle-only follow-up report, unchanged)
